In [1]:
import pickle
from collections import OrderedDict
import os
import torch
import numpy as np
import pandas as pd
import sklearn
import random
import h5py
from sklearn.model_selection import GroupKFold
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scanpy as sc
import anndata as ad
from textwrap import shorten

In [2]:
adata = sc.read('./Data/FinalData/adataAfterClean.h5ad')

In [3]:
h5_dst = (
    "./Code/other_models/TranSiGen/data/Meisheng_used_data/"
    "processed_data_id.h5"
)

In [4]:
with h5py.File(h5_dst, "r") as f:
    sig_raw = np.asarray(f["sig"])          # bytes or str
    if isinstance(sig_raw[0], (bytes, np.bytes_)):
        sigs_str = sig_raw.astype(str)
    else:
        sigs_str = sig_raw                  # already str

n_samples = sigs_str.size
print("Samples:", n_samples)

Samples: 836649


In [5]:
split_cols = [
    f"{kind}_split{i}"
    for i in range(1, 6)
    for kind in ("random", "cell", "drug")
]  # ['random_split1', 'cell_split1', ... 'drug_split5']

# DataFrame whose index order matches sigs_str
split_df = adata.obs.loc[sigs_str, split_cols]

# Sanity check
assert (split_df.index.values == sigs_str).all(), "Index order mismatch"

In [6]:
split_df

,random_split1,cell_split1,drug_split1,random_split2,cell_split2,drug_split2,random_split3,cell_split3,drug_split3,random_split4,cell_split4,drug_split4,random_split5,cell_split5,drug_split5
index,,,,,,,,,,,,,,,
REP.A001_A375_24H_X1_B22:B13-2,train,valid,train,test,train,train,train,train,train,test,train,train,train,train,test
REP.A001_A375_24H_X1_B22:B14-2,train,valid,train,train,train,train,train,train,train,train,train,train,test,train,test
REP.A001_A375_24H_X1_B22:B15-2,train,valid,train,train,train,train,train,train,train,train,train,train,test,train,test
REP.A001_A375_24H_X1_B22:B16-2,train,valid,train,train,train,train,train,train,train,train,train,train,train,train,test
REP.A001_A375_24H_X1_B22:B17-2,train,valid,train,train,train,train,train,train,train,test,train,train,train,train,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PCLB003_PC3_24H_X3_B13:P20-1,train,train,train,train,train,train,train,train,train,train,train,train,train,train,train
PCLB003_PC3_24H_X3_B13:P21-1,train,train,train,test,train,train,train,train,train,train,train,train,valid,train,train
PCLB003_PC3_24H_X3_B13:P22-1,train,train,train,valid,train,train,test,train,train,test,train,train,train,train,train


In [7]:
with h5py.File(h5_dst, "a") as f:                     # 'a' = append mode
    for col in split_cols:
        col_vals = split_df[col].astype(str).to_numpy()      # array of str
        max_len  = max(len(s) for s in col_vals) or 1
        col_bytes = np.asarray(
            [s.encode("utf-8") for s in col_vals],
            dtype=f"S{max_len}"
        )

        if col in f:          # overwrite if it already exists
            del f[col]
        f.create_dataset(col, data=col_bytes)

print("Appended split columns to", h5_dst)

Appended split columns to /work/users/m/e/meisheng/Dissertation/TranSiGen/data/Meisheng_used_data/processed_data_id_halfway.h5


In [8]:
'''
sanity check
'''
with h5py.File(h5_dst, "r") as f:
    for col in ["random_split1", "cell_split1", "drug_split1"]:
        print(col, "→", f[col][:5])     # show first 5 samples

random_split1 → [b'train' b'train' b'train' b'train' b'train']
cell_split1 → [b'valid' b'valid' b'valid' b'valid' b'valid']
drug_split1 → [b'train' b'train' b'train' b'train' b'train']


In [9]:
def load_from_HDF(fname):
    """Load data from a HDF5 file to a dictionary."""
    data = dict()
    with h5py.File(fname, 'r') as f:
        for key in f:
            data[key] = np.asarray(f[key])
            if isinstance(data[key][0], np.bytes_):
                data[key] = data[key].astype(str)
    return data

data = load_from_HDF(h5_dst)

In [10]:
data

{'LINCS_index': array([     0,      1,      2, ..., 836646, 836647, 836648]),
 'canonical_smiles': array([   0,    0,    0, ..., 1419, 1419, 1419]),
 'cell_split1': array(['valid', 'valid', 'valid', ..., 'train', 'train', 'train'],
       dtype='<U5'),
 'cell_split2': array(['train', 'train', 'train', ..., 'train', 'train', 'train'],
       dtype='<U5'),
 'cell_split3': array(['train', 'train', 'train', ..., 'train', 'train', 'train'],
       dtype='<U5'),
 'cell_split4': array(['train', 'train', 'train', ..., 'train', 'train', 'train'],
       dtype='<U5'),
 'cell_split5': array(['train', 'train', 'train', ..., 'train', 'train', 'train'],
       dtype='<U5'),
 'cid': array(['A375', 'A375', 'A375', ..., 'PC3', 'PC3', 'PC3'], dtype='<U8'),
 'drug_split1': array(['train', 'train', 'train', ..., 'train', 'train', 'train'],
       dtype='<U5'),
 'drug_split2': array(['train', 'train', 'train', ..., 'train', 'train', 'train'],
       dtype='<U5'),
 'drug_split3': array(['train', 'train', 't

In [11]:
# ── imports & regex helper ─────────────────────────────────────────────
import re, pandas as pd
from pprint import pprint      # optional, nice console print

# assume `data = load_from_HDF(h5_dst)` already ran
# ----------------------------------------------------------------------

# 1) grab every key matching "<type>_split<1-5>"
split_keys = [k for k in data if re.match(r'^(random|cell|drug)_split[1-5]$', k)]

# 2) collect counts in a dict-of-dicts → {col → {'train': n, 'test': n, 'valid': n}}
counts = {}
for col in split_keys:
    uniq, cnt = np.unique(data[col], return_counts=True)
    counts[col] = {u: int(c) for u, c in zip(uniq, cnt)}   # make int for DataFrame

# 3) DataFrame for a nice view (rows = split columns, cols = split label)
df_counts = pd.DataFrame(counts).T.fillna(0).astype(int)   # NA→0, ensure int
df_counts = df_counts[["train", "test", "valid"]]          # ordered columns

# 4) display
display(df_counts)            # Jupyter notebook view
# or, for plain console:
# pprint(df_counts.to_dict(orient="index"))


,train,test,valid
cell_split1,633012,12510,191127
cell_split2,730869,25742,80038
cell_split3,619257,156258,61134
cell_split4,671357,137725,27567
cell_split5,590631,86370,159648
drug_split1,660833,85539,90277
drug_split2,667131,86935,82583
drug_split3,670367,78097,88185
drug_split4,675250,77889,83510
drug_split5,661871,80260,94518


In [12]:
# ------------------------------------------------------------
# 1) identify the cell_split columns
# ------------------------------------------------------------
cell_split_cols = [k for k in data if re.match(r'^cell_split[1-5]$', k)]
print("Cell-split columns:", cell_split_cols)

# ------------------------------------------------------------
# 2) gather unique cid sets for each split label
# ------------------------------------------------------------
split_summary = {}     # {cell_splitX → {label → set(cids)}}

for col in cell_split_cols:
    summary = {}
    for label in ["train", "test", "valid"]:
        mask = data[col] == label
        summary[label] = set(np.unique(data["cid"][mask]))
    split_summary[col] = summary

# ------------------------------------------------------------
# 3) pretty-print the results
# ------------------------------------------------------------
pprint(split_summary, depth=3, compact=True)

Cell-split columns: ['cell_split1', 'cell_split2', 'cell_split3', 'cell_split4', 'cell_split5']
{'cell_split1': {'test': {'CD34', 'JURKAT', 'MNEU.E', 'NCIH2073', 'NCIH508',
                          'OV7', 'SKL', 'THP1', 'WSUDLCL2'},
                 'train': {'A549', 'A673', 'AGS', 'ASC', 'ASC.C', 'BT20',
                           'CL34', 'CORL23', 'COV644', 'DV90', 'EFO27',
                           'FIBRNPC', 'H1299', 'HA1E', 'HCC15', 'HCC515',
                           'HCT116', 'HEC108', 'HEK293T', 'HELA', 'HEPG2',
                           'HL60', 'HS27A', 'HS578T', 'HT29', 'HUES3', 'HUH7',
                           'HUVEC', 'JHUEM2', 'LNCAP', 'LOVO', 'MCF10A', 'MCF7',
                           'MDAMB231', 'NCIH1694', 'NCIH1836', 'NCIH596', 'NEU',
                           'NKDBA', 'NOMO1', 'NPC', 'NPC.CAS9', 'NPC.TAK',
                           'PC3', 'PHH', 'PL21', 'RKO', 'RMGI', 'SKB', 'SKBR3',
                           'SKLU1', 'SKM1', 'SKMEL1', 'SKMEL28', 'SNGM',
  